# Avaliação de Previsões (Janela Deslizante)

Este notebook carrega previsões em CSV na pasta `previsoes/` e executa:

1. **Métricas por janela e série**: MAE e MSE dos resíduos.
2. **Box-plots**: por série/canal e agregado global.
3. **Gráficos de séries**: observado (preto) vs previsto (vermelho).

As previsões são avaliadas em três paradigmas:
- `first`: primeira vez que o ponto foi previsto.
- `median`: mediana de todas as previsões para o mesmo ponto.
- `last`: última vez que o ponto foi previsto.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

# Caminho padrão das previsões (CSV)
PREDICOES_DIR = Path("previsoes")

# Nomes de colunas aceitos (fallback para formatos diferentes)
COL_ALIASES = {
    "serie": ["serie", "series", "canal", "channel", "id_serie"],
    "timestamp": ["timestamp", "data", "date", "ds", "t"],
    "observado": ["observado", "real", "y_true", "actual", "y"],
    "previsto": ["previsto", "pred", "y_pred", "forecast", "yhat"],
    "lote": ["lote", "batch", "janela", "window", "rolling_window"],
}


def _normalize_column_names(df: pd.DataFrame) -> pd.DataFrame:
    rename_map = {}
    lower_to_original = {c.lower(): c for c in df.columns}

    for canonical, aliases in COL_ALIASES.items():
        for alias in aliases:
            if alias.lower() in lower_to_original:
                rename_map[lower_to_original[alias.lower()]] = canonical
                break

    out = df.rename(columns=rename_map).copy()
    required = ["serie", "timestamp", "observado", "previsto", "lote"]
    missing = [c for c in required if c not in out.columns]
    if missing:
        raise ValueError(
            "Colunas obrigatórias ausentes. Esperado ao menos aliases para: "
            f"{required}. Faltando: {missing}."
        )

    out["timestamp"] = pd.to_datetime(out["timestamp"], errors="coerce")
    if out["timestamp"].isna().any():
        raise ValueError("Não foi possível converter todos os timestamps para datetime.")

    return out


def load_prediction_csvs(predicoes_dir: Path = PREDICOES_DIR) -> pd.DataFrame:
    files = sorted(predicoes_dir.glob("*.csv"))
    if not files:
        raise FileNotFoundError(f"Nenhum CSV encontrado em: {predicoes_dir.resolve()}")

    dfs = []
    for file in files:
        df = pd.read_csv(file)
        df = _normalize_column_names(df)
        df["arquivo_origem"] = file.name
        dfs.append(df)

    data = pd.concat(dfs, ignore_index=True)
    data = data.sort_values(["serie", "timestamp", "lote"]).reset_index(drop=True)
    return data


def collapse_predictions_by_paradigm(data: pd.DataFrame) -> pd.DataFrame:
    ordered = data.sort_values(["serie", "timestamp", "lote"]).copy()

    grp = ordered.groupby(["serie", "timestamp"], as_index=False)

    first_df = grp.first()[["serie", "timestamp", "observado", "previsto"]].copy()
    first_df["paradigma"] = "first"

    median_df = grp.agg({"observado": "first", "previsto": "median"})
    median_df["paradigma"] = "median"

    last_df = grp.last()[["serie", "timestamp", "observado", "previsto"]].copy()
    last_df["paradigma"] = "last"

    collapsed = pd.concat([first_df, median_df, last_df], ignore_index=True)
    collapsed["residuo"] = collapsed["observado"] - collapsed["previsto"]
    collapsed["erro_abs"] = collapsed["residuo"].abs()
    collapsed["erro_sq"] = collapsed["residuo"] ** 2
    return collapsed.sort_values(["paradigma", "serie", "timestamp"]).reset_index(drop=True)


def metrics_per_window_and_series(collapsed: pd.DataFrame, freq="M") -> pd.DataFrame:
    work = collapsed.copy()
    work["janela"] = work["timestamp"].dt.to_period(freq).dt.to_timestamp()

    per_series = (
        work.groupby(["paradigma", "serie", "janela"], as_index=False)
        .agg(MAE=("erro_abs", "mean"), MSE=("erro_sq", "mean"))
        .sort_values(["paradigma", "serie", "janela"])
    )

    aggregated = (
        per_series.groupby(["paradigma", "janela"], as_index=False)
        .agg(MAE=("MAE", "mean"), MSE=("MSE", "mean"))
        .assign(serie="AGREGADO")
    )

    out = pd.concat([per_series, aggregated], ignore_index=True)
    return out.sort_values(["paradigma", "serie", "janela"]).reset_index(drop=True)


def plot_metrics_lines(metrics_df: pd.DataFrame, metric: str, paradigm: str):
    df = metrics_df[(metrics_df["paradigma"] == paradigm)].copy()
    if df.empty:
        print(f"Sem dados para paradigma={paradigm}")
        return

    fig, ax = plt.subplots(figsize=(12, 5))
    for serie, g in df.groupby("serie"):
        ax.plot(g["janela"], g[metric], marker="o", linewidth=1.5, label=serie)

    ax.set_title(f"{metric} por janela - paradigma: {paradigm}")
    ax.set_xlabel("Janela")
    ax.set_ylabel(metric)
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()


def boxplots_metrics(collapsed: pd.DataFrame):
    # Erro absoluto (MAE ponto-a-ponto) e erro quadrático (MSE ponto-a-ponto)
    long_df = pd.concat([
        collapsed[["serie", "paradigma", "erro_abs"]].rename(columns={"erro_abs": "valor"}).assign(metrica="MAE"),
        collapsed[["serie", "paradigma", "erro_sq"]].rename(columns={"erro_sq": "valor"}).assign(metrica="MSE"),
    ], ignore_index=True)

    agg = long_df.groupby(["paradigma", "metrica"], as_index=False)["valor"].mean()
    agg["serie"] = "AGREGADO"
    plot_df = pd.concat([long_df, agg], ignore_index=True)

    for metrica in ["MAE", "MSE"]:
        fig, ax = plt.subplots(figsize=(14, 6))
        sub = plot_df[plot_df["metrica"] == metrica]
        sns.boxplot(data=sub, x="serie", y="valor", hue="paradigma", ax=ax)
        ax.set_title(f"Box-plot {metrica} por série/canal e agregado")
        ax.set_xlabel("Série")
        ax.set_ylabel(metrica)
        ax.tick_params(axis="x", rotation=45)
        plt.tight_layout()
        plt.show()


def plot_series_real_vs_pred(collapsed: pd.DataFrame, paradigm: str = "median"):
    df = collapsed[collapsed["paradigma"] == paradigm].copy()
    for serie, g in df.groupby("serie"):
        g = g.sort_values("timestamp")
        fig, ax = plt.subplots(figsize=(12, 4))
        ax.plot(g["timestamp"], g["observado"], color="black", linewidth=1.8, label="Observado")
        ax.plot(g["timestamp"], g["previsto"], color="red", linewidth=1.5, label="Previsto")
        ax.set_title(f"Série: {serie} | Paradigma: {paradigm}")
        ax.set_xlabel("Tempo")
        ax.set_ylabel("Valor")
        ax.legend()
        plt.tight_layout()
        plt.show()

In [ ]:
# Execução principal
data = load_prediction_csvs(PREDICOES_DIR)
collapsed = collapse_predictions_by_paradigm(data)
metrics = metrics_per_window_and_series(collapsed, freq="M")

print("Amostra das métricas por janela/série:")
display(metrics.head(20))

for paradigm in ["first", "median", "last"]:
    print(f"\n=== Paradigma: {paradigm} ===")
    display(metrics[metrics["paradigma"] == paradigm].head(20))
    plot_metrics_lines(metrics, metric="MAE", paradigm=paradigm)
    plot_metrics_lines(metrics, metric="MSE", paradigm=paradigm)

boxplots_metrics(collapsed)

# gráficos de série (observado preto vs previsto vermelho)
for paradigm in ["first", "median", "last"]:
    plot_series_real_vs_pred(collapsed, paradigm=paradigm)